# TripGenie — Tourism Data Preprocessing

## 1. Purpose and preprocessing policy
Prepare the supplied tourism catalogue for future TF-IDF content matching and personalized recommendation filters. The source is `data/india_tourism_dataset.json`, described by the project owner as the Kaggle tourism dataset; a Kaggle URL, license, and independent provenance are not present in the file and are not inferred here.

This notebook preserves every record and original ID. Duplicate destination names are **investigated and flagged, never automatically removed or merged**. Keep descriptive content and conflicting facts available for later review. The original JSON is read only; outputs go to `data/processed/`.

Missing text becomes empty text only in derived text features; unknown numeric values remain null with missing indicators. Unknown distances are not zero. No vectorizer, scaler, imputer, or recommendation model is fitted. Statistical preprocessing must be fitted on a future training split to avoid leakage.

## 2. Load the source and inspect its schema
Run all cells in order with Python 3, pandas, and NumPy. Validation execution additionally uses `nbformat`, `nbclient`, and `ipykernel`. The repository is resolved from the working directory or its parents.

In [1]:
from pathlib import Path
from itertools import combinations
import hashlib
import json
import re
import unicodedata
import platform
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def observe(text):
    display(Markdown("**Observations.** " + text))

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data" / "india_tourism_dataset.json").is_file()), None)
if root is None:
    raise FileNotFoundError("Run this notebook from within the TripGenie repository.")
source_path = root / "data" / "india_tourism_dataset.json"
source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
with source_path.open(encoding="utf-8") as handle:
    records = json.load(handle)
assert isinstance(records, list) and records and all(isinstance(r, dict) for r in records)
raw = pd.DataFrame(records)
assert raw.id.notna().all() and raw.id.is_unique, "IDs must uniquely identify preserved entries."
print(f"Source: data/{source_path.name}; {raw.shape[0]} entries × {raw.shape[1]} top-level fields")
display(pd.DataFrame({"field": raw.columns, "dtype": raw.dtypes.astype(str).values}))
display(raw[["id", "destination_name", "state", "coordinates", "trip_types", "budget_category"]].head(3))

Source: data/india_tourism_dataset.json; 100 entries × 54 top-level fields


,field,dtype
0,id,int64
1,state,str
2,district,str
3,region,str
4,coordinates,object
5,altitude_m,int64
6,popularity_score,int64
7,accessibility,str
8,nearest_airport,object
9,nearest_railway_station,object


,id,destination_name,state,coordinates,trip_types,budget_category
0,1,Goa,Goa,"{'latitude': 15.3, 'longitude': 73.8}","[Beach, Cultural, Food, Nightlife, Wellness, P...","{'accommodation_range': [800, 1500], 'food_ran..."
1,2,Ladakh (Leh),Ladakh,"{'latitude': 34.16, 'longitude': 77.58}","[Adventure, Nature, Cultural, Religious, Photo...","{'accommodation_range': [600, 1500], 'food_ran..."
2,3,Jaipur,Rajasthan,"{'latitude': 26.9124, 'longitude': 75.7873}","[Heritage, Cultural, Photography, Shopping, Food]","{'accommodation_range': [700, 1400], 'food_ran..."


## 3. Investigate repeated destinations before transformation
Group by normalized destination name **and state/location**, using Unicode normalization, case folding, and collapsed whitespace only. This is a conservative review key, not a geographic entity ID: synonyms and alternative names may still be missed.

Compare every pair in a group using original field values, excluding only the record ID. The report identifies differing fields and retains their values for inspection. No similarity threshold or automatic merge rule is applied. Conflicting permits, costs, and durations must not be silently combined.

In [2]:
def identity_key(value):
    return " ".join(unicodedata.normalize("NFKC", str(value)).casefold().split())

review_keys = pd.Series([identity_key(r["destination_name"]) + " | " + identity_key(r["state"])
                        for r in records], index=raw.index)
duplicate_reports = []
for key, indexes in review_keys.groupby(review_keys).groups.items():
    indexes = list(indexes)
    if len(indexes) < 2:
        continue
    for left_index, right_index in combinations(indexes, 2):
        left, right = records[left_index], records[right_index]
        differences = {field: {str(left["id"]): left.get(field), str(right["id"]): right.get(field)}
                       for field in sorted(set(left) | set(right))
                       if field != "id" and left.get(field) != right.get(field)}
        duplicate_reports.append({"review_key": key, "left_id": left["id"], "right_id": right["id"],
                                  "differing_field_count": len(differences), "differences": differences})
duplicate_table = pd.DataFrame([{k: v for k, v in row.items() if k != "differences"}
                                | {"differing_fields": list(row["differences"])}
                                for row in duplicate_reports])
with pd.option_context("display.max_colwidth", 150):
    display(duplicate_table)
duplicate_mask = review_keys.duplicated(keep=False)
display(raw.loc[duplicate_mask, ["id", "destination_name", "state", "region", "popularity_score",
                               "minimum_days", "ideal_days", "maximum_days", "permits_required"]]
        .sort_values(["destination_name", "id"]))
observe(f"Found {review_keys[duplicate_mask].nunique()} repeated destination/location groups involving "
        f"{duplicate_mask.sum()} records. Compared {len(duplicate_reports)} pairs; "
        f"{sum(bool(r['differences']) for r in duplicate_reports)} pairs differ beyond ID. "
        "All entries are retained; the detailed report preserves conflicting values for review.")

,review_key,left_id,right_id,differing_field_count,differing_fields
0,coorg (kodagu) | karnataka,8,53,40,"[accommodation_types, activities_available, atm_availability, average_temperature, avoid_seasons, best_seasons, budget_category, coordinates, data..."
1,gokarna | karnataka,25,71,38,"[accommodation_types, activities_available, atm_availability, average_temperature, avoid_seasons, best_seasons, budget_category, coordinates, data..."
2,havelock island (swaraj dweep) | andaman and nicobar islands,31,61,41,"[accommodation_types, activities_available, atm_availability, average_temperature, avoid_seasons, best_seasons, budget_category, coordinates, data..."
3,neil island (shaheed dweep) | andaman and nicobar islands,32,62,42,"[accommodation_types, activities_available, atm_availability, average_temperature, avoid_seasons, best_seasons, budget_category, coordinates, data..."
4,spiti valley | himachal pradesh,16,74,42,"[accommodation_types, activities_available, atm_availability, average_temperature, avoid_seasons, best_seasons, budget_category, coordinates, data..."
5,tawang | arunachal pradesh,14,73,46,"[accommodation_types, activities_available, altitude_m, atm_availability, average_temperature, avoid_seasons, best_seasons, budget_category, coord..."
6,varanasi (kashi) | uttar pradesh,4,39,39,"[accommodation_types, activities_available, altitude_m, atm_availability, average_temperature, avoid_seasons, best_seasons, budget_category, data_..."


,id,destination_name,state,region,popularity_score,minimum_days,ideal_days,maximum_days,permits_required
7,8,Coorg (Kodagu),Karnataka,South India,7,2.0,3.0,4,False
52,53,Coorg (Kodagu),Karnataka,South India,8,2.0,3.0,5,False
24,25,Gokarna,Karnataka,South India,6,2.0,3.0,5,False
70,71,Gokarna,Karnataka,South India,8,2.0,3.0,5,False
30,31,Havelock Island (Swaraj Dweep),Andaman and Nicobar Islands,Island Territory,9,2.0,3.0,5,False
60,61,Havelock Island (Swaraj Dweep),Andaman and Nicobar Islands,Island Territory,9,2.0,3.0,5,True
31,32,Neil Island (Shaheed Dweep),Andaman and Nicobar Islands,Island Territory,7,1.0,2.0,3,False
61,62,Neil Island (Shaheed Dweep),Andaman and Nicobar Islands,Island Territory,8,1.0,2.0,3,True
15,16,Spiti Valley,Himachal Pradesh,North India,6,5.0,7.0,12,True
73,74,Spiti Valley,Himachal Pradesh,North India,8,6.0,7.0,12,True


**Observations.** Found 7 repeated destination/location groups involving 14 records. Compared 7 pairs; 7 pairs differ beyond ID. All entries are retained; the detailed report preserves conflicting values for review.

## 4. Missing values and nested JSON normalization
Recursively standardize explicit nulls, blank strings, and `data_not_available` to null. Record their paths in an audit; meaningful phrases such as “Not required” stay intact. Clean scalar whitespace without changing letter case or punctuation in retained descriptions. Lists remain native arrays, with missing items excluded from derived content.

Flatten fixed dictionaries into dotted fields for coordinates, transport, temperatures, and cost ranges. Keep `ideal_for_why` as a dictionary because its audience keys vary. Temperature strings are not forced into numeric values.

In [3]:
missing_audit = []
def clean_value(value, record_id, path=""):
    if value is None or (isinstance(value, str) and
                         (not value.strip() or value.strip().casefold() == "data_not_available")):
        missing_audit.append({"id": record_id, "field_path": path, "reason": "null/blank/sentinel"})
        return None
    if isinstance(value, dict):
        if not value:
            missing_audit.append({"id": record_id, "field_path": path, "reason": "empty dictionary"})
        return {k: clean_value(v, record_id, f"{path}.{k}" if path else k) for k, v in value.items()}
    if isinstance(value, list):
        if not value:
            missing_audit.append({"id": record_id, "field_path": path, "reason": "empty list"})
        cleaned = [clean_value(v, record_id, f"{path}.{i}") for i, v in enumerate(value)]
        return [v for v in cleaned if v is not None]
    if isinstance(value, str):
        return " ".join(unicodedata.normalize("NFKC", value).split())
    return value

cleaned_records = [clean_value(r, r["id"]) for r in records]
processed = pd.json_normalize([{k: v for k, v in r.items() if k != "ideal_for_why"}
                              for r in cleaned_records], sep=".")
processed["ideal_for_why"] = [r.get("ideal_for_why", {}) for r in cleaned_records]
# Preserve exact display names; normalize only derived identity and text features.
processed["destination_name"] = raw.destination_name
processed["destination_review_key"] = review_keys
processed["duplicate_destination_flag"] = duplicate_mask
processed["duplicate_group_size"] = review_keys.map(review_keys.value_counts()).astype(int)
processed["region_original"] = raw.region
processed["region"] = processed.region.replace({"North East India": "Northeast India"})
display(pd.DataFrame(missing_audit))
display(processed.isna().sum().loc[lambda x: x > 0].rename("null_count").to_frame())
observe(f"Standardized {len(missing_audit)} missing-like values or containers. "
        "Nulls remain explicit in retained fields; region spelling is harmonized with the original label retained.")

,id,field_path,reason
0,2,average_temperature.monsoon,null/blank/sentinel
1,31,nearest_railway_station.distance_km,null/blank/sentinel
2,32,nearest_railway_station.distance_km,null/blank/sentinel
3,49,nearest_railway_station.distance_km,null/blank/sentinel
4,50,nearest_railway_station.distance_km,null/blank/sentinel
5,61,nearest_railway_station.distance_km,null/blank/sentinel
6,62,nearest_railway_station.distance_km,null/blank/sentinel
7,63,nearest_railway_station.distance_km,null/blank/sentinel
8,88,nearest_railway_station.distance_km,null/blank/sentinel


,null_count
nearest_railway_station.distance_km,8
average_temperature.monsoon,1


**Observations.** Standardized 9 missing-like values or containers. Nulls remain explicit in retained fields; region spelling is harmonized with the original label retained.

## 5. Structured numeric and budget features
Keep coordinates, durations, ratings, altitude, and transport distances numeric. Add missing indicators without statistical imputation. Reject malformed or reversed cost ranges so invalid values cannot silently reach downstream filters. Split each tier/component range into lower and upper endpoints, and derive daily midpoint and width.

Check supplied totals against component sums. Preserve the supplied ranges and flag mismatches rather than repairing uncertain source values. Currency and per-person assumptions are undocumented. Permits remain boolean; an unknown value would remain null rather than becoming false.

In [4]:
numeric_fields = ["coordinates.latitude", "coordinates.longitude", "altitude_m", "popularity_score",
                  "safety_rating", "minimum_days", "ideal_days", "maximum_days",
                  "nearest_airport.distance_km", "nearest_railway_station.distance_km",
                  "nearest_major_city.distance_km"]
for field in numeric_fields:
    original = processed[field]
    converted = pd.to_numeric(original, errors="coerce")
    assert not (original.notna() & converted.isna()).any(), f"Unexpected numeric value in {field}"
    processed[field] = converted
    processed[field + "_missing"] = converted.isna()

tiers = ["budget_category", "mid_range_category", "luxury_category"]
components = ["accommodation", "food", "activities", "local_transport"]
budget_audit = []
for tier in tiers:
    for component in [*components, "total_daily"]:
        field = f"{tier}.{component}_range"
        valid = processed[field].map(lambda v: isinstance(v, list) and len(v) == 2
                                    and all(isinstance(x, (int, float)) and not isinstance(x, bool)
                                            and np.isfinite(x) for x in v) and 0 <= v[0] <= v[1])
        assert valid.all(), f"Malformed cost ranges in {field}; inspect before exporting."
        processed[f"{tier}.{component}_low"] = processed[field].map(lambda v: v[0])
        processed[f"{tier}.{component}_high"] = processed[field].map(lambda v: v[1])
    low, high = processed[f"{tier}.total_daily_low"], processed[f"{tier}.total_daily_high"]
    processed[f"{tier}.daily_midpoint"] = (low + high) / 2
    processed[f"{tier}.daily_range_width"] = high - low
    matches = pd.Series(True, index=processed.index)
    for endpoint in ["low", "high"]:
        total = processed[[f"{tier}.{c}_{endpoint}" for c in components]].sum(axis=1)
        matches &= np.isclose(processed[f"{tier}.total_daily_{endpoint}"], total)
    processed[f"{tier}.total_components_mismatch"] = ~matches
    budget_audit.append({"tier": tier, "mismatched_entries": int((~matches).sum())})
display(pd.DataFrame(budget_audit))
display(processed[["id", "destination_name", "nearest_railway_station.distance_km",
                   "nearest_railway_station.distance_km_missing", "budget_category.daily_midpoint"]].head())
observe(f"Railway distances remain unknown for {processed['nearest_railway_station.distance_km_missing'].sum()} entries. "
        f"Flagged {sum(row['mismatched_entries'] for row in budget_audit)} entry–tier total/component discrepancies; "
        "no source cost totals were replaced.")

,tier,mismatched_entries
0,budget_category,0
1,mid_range_category,3
2,luxury_category,11


,id,destination_name,nearest_railway_station.distance_km,nearest_railway_station.distance_km_missing,budget_category.daily_midpoint
0,1,Goa,35.0,False,2100.0
1,2,Ladakh (Leh),420.0,False,2450.0
2,3,Jaipur,3.0,False,1950.0
3,4,Varanasi (Kashi),2.0,False,1800.0
4,5,Agra,3.0,False,2100.0


**Observations.** Railway distances remain unknown for 8 entries. Flagged 14 entry–tier total/component discrepancies; no source cost totals were replaced.

## 6. Clean text and prepare multi-label preferences
Derived text uses Unicode normalization, case folding, underscore-to-space conversion, punctuation removal, and collapsed whitespace. Keep Unicode letters and numbers. Original descriptive fields remain available; there is no stemming or stopword removal that might discard meaning. A future TF-IDF vectorizer can choose those settings explicitly.

Clean list-valued preference fields item by item, remove empty items, and deduplicate only tags **within a record**, preserving order. Apply a small, explicit alias dictionary for obvious wording variants. Do not merge more specific interests into broad categories, or map “Family with kids” to “Family.” Audience explanations are retained separately.

In [5]:
def clean_text(value):
    if value is None or (isinstance(value, (float, np.floating)) and np.isnan(value)):
        return ""
    if isinstance(value, dict):
        return " ".join(part for v in value.values() if (part := clean_text(v)))
    if isinstance(value, list):
        return " ".join(part for v in value if (part := clean_text(v)))
    text = unicodedata.normalize("NFKC", str(value)).casefold().replace("_", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())

trip_aliases = {"cultural": "culture", "waterfall": "waterfalls",
                "short treks": "short trek", "backpacker": "backpacking"}
audience_aliases = {"couples": "couple", "families": "family", "families with kids": "family with kids",
                    "history buffs": "history buff", "slow travelers": "slow travellers"}
preference_fields = ["trip_types", "ideal_for", "best_seasons", "avoid_seasons", "activities_available",
                     "primary_attractions", "hidden_gems", "accommodation_types", "language_spoken"]
for field in preference_fields:
    assert processed[field].map(lambda v: isinstance(v, list)).all(), f"Expected arrays in {field}"
    aliases = trip_aliases if field == "trip_types" else audience_aliases if field == "ideal_for" else {}
    def clean_tags(values):
        tags = [aliases.get(text, text) for v in values if (text := clean_text(v))]
        return list(dict.fromkeys(tags))
    processed[field + "_clean"] = processed[field].map(clean_tags)

display(pd.DataFrame({"raw_trip_type": sorted({t for r in records for t in r['trip_types']})})
        .assign(clean_trip_type=lambda x: x.raw_trip_type.map(lambda v: trip_aliases.get(clean_text(v), clean_text(v))))
        .head(20))
display(processed[["id", "trip_types", "trip_types_clean", "ideal_for_clean"]].head())
observe(f"Trip tags change from {len({t for r in records for t in r['trip_types']})} raw labels to "
        f"{len({t for tags in processed.trip_types_clean for t in tags})} normalized labels. "
        "Arrays are preserved for future multi-hot encoding and preference matching.")

,raw_trip_type,clean_trip_type
0,Adventure,adventure
1,Ashram,ashram
2,Backpacker,backpacking
3,Backpacking,backpacking
4,Backwaters,backwaters
5,Beach,beach
6,Birding,birding
7,Border_region,border region
8,Buddhist_circuit,buddhist circuit
9,Café_town,café town


,id,trip_types,trip_types_clean,ideal_for_clean
0,1,"[Beach, Cultural, Food, Nightlife, Wellness, P...","[beach, culture, food, nightlife, wellness, ph...","[solo, couple, family with kids, family with e..."
1,2,"[Adventure, Nature, Cultural, Religious, Photo...","[adventure, nature, culture, religious, photog...","[adventure seekers, photographers, trekking en..."
2,3,"[Heritage, Cultural, Photography, Shopping, Food]","[heritage, culture, photography, shopping, food]","[family, couple, history buff, first time indi..."
3,4,"[Religious, Spiritual, Cultural, Photography]","[religious, spiritual, culture, photography]","[pilgrims, spiritual seekers, photographers, c..."
4,5,"[Heritage, Photography, Short_break]","[heritage, photography, short break]","[first time visitors, heritage lovers, day tri..."


**Observations.** Trip tags change from 103 raw labels to 99 normalized labels. Arrays are preserved for future multi-hot encoding and preference matching.

## 7. Build TF-IDF input documents and a feature contract
Combine destination names, geography, interests, attractions, activities, experiences, culture, and food into one clean document per entry. Exclude IDs, source tokens, popularity, and prices from semantic text. Seasonal advice, permits, safety, budgets, and durations remain separate structured fields for constraints and explanations.

No vectors or similarity scores are computed. `tfidf_text` is a baseline input; future work can compare separate field representations and weights. Keep `destination_review_key` for grouping duplicate candidates in evaluation splits to reduce leakage. A flag is not a final entity-resolution decision.

In [6]:
text_fields = ["destination_name", "state", "region", "trip_types_clean", "primary_attractions_clean",
               "activities_available_clean", "unique_experiences", "hidden_gems_clean", "local_culture", "food_scene"]
processed["tfidf_text"] = processed[text_fields].apply(
    lambda row: " ".join(part for v in row if (part := clean_text(v))), axis=1)
processed["tfidf_word_count"] = processed.tfidf_text.str.split().map(len)
processed["tfidf_text_missing"] = processed.tfidf_text.eq("")
feature_contract = {
    "semantic_text": {"column": "tfidf_text", "source_fields": text_fields},
    "identity_and_review": ["id", "destination_name", "destination_review_key", "duplicate_destination_flag", "duplicate_group_size"],
    "numeric": numeric_fields + [f"{tier}.{feature}" for tier in tiers
                                 for feature in ["total_daily_low", "total_daily_high", "daily_midpoint", "daily_range_width"]],
    "categorical": ["state", "region", "accessibility"],
    "multi_label": [f"{field}_clean" for field in preference_fields],
    "boolean_constraints": ["permits_required"],
    "missing_indicators": [field + "_missing" for field in numeric_fields] + ["tfidf_text_missing"],
    "quality_flags": [f"{tier}.total_components_mismatch" for tier in tiers],
    "future_training_note": "Fit vectorizers, statistical imputers, encoders, and scalers on training data only. No fitting occurs here."
}
display(processed[["id", "destination_name", "tfidf_text", "tfidf_word_count"]].head(3))
display(processed.tfidf_word_count.describe().to_frame())
observe(f"Prepared {len(processed)} documents; {processed.tfidf_text_missing.sum()} are empty. "
        f"Median document length is {processed.tfidf_word_count.median():g} whitespace-delimited words. "
        "Documents are linked to original record IDs, including all duplicate destination candidates.")

,id,destination_name,tfidf_text,tfidf_word_count
0,1,Goa,goa goa west india beach culture food nightlif...,79
1,2,Ladakh (Leh),ladakh leh ladakh north india adventure nature...,83
2,3,Jaipur,jaipur rajasthan north india heritage culture ...,79


,tfidf_word_count
count,100.000000
mean,101.490000
std,17.369133
min,62.000000
25%,89.500000
50%,102.000000
75%,114.250000
max,162.000000


**Observations.** Prepared 100 documents; 0 are empty. Median document length is 102 whitespace-delimited words. Documents are linked to original record IDs, including all duplicate destination candidates.

## 8. Validate before export
Check row and ID preservation, original names, duplicate flags, native lists, numeric ranges, duration ordering, coordinate bounds, boolean permits, and nonempty semantic documents. Verify known text-cleaning cases without fitting a model. Checks validate transformations and internal consistency; they do not verify real-world travel facts.

In [7]:
assert len(processed) == len(records)
assert processed.id.tolist() == raw.id.tolist() and processed.id.is_unique
assert processed.destination_name.tolist() == raw.destination_name.tolist()
assert processed.duplicate_destination_flag.tolist() == duplicate_mask.tolist()
assert processed.duplicate_group_size.sum() >= len(processed)
assert processed["coordinates.latitude"].dropna().between(-90, 90).all()
assert processed["coordinates.longitude"].dropna().between(-180, 180).all()
for field in ["minimum_days", "ideal_days", "maximum_days"]:
    assert processed[field].dropna().gt(0).all()
assert (processed.minimum_days <= processed.ideal_days).all()
assert (processed.ideal_days <= processed.maximum_days).all()
for field in numeric_fields:
    assert pd.api.types.is_numeric_dtype(processed[field])
    assert processed[field + "_missing"].equals(processed[field].isna())
for field in numeric_fields[8:]:
    assert processed[field].dropna().ge(0).all()
assert processed.permits_required.map(lambda v: v is None or isinstance(v, (bool, np.bool_))).all()
for field in preference_fields:
    assert processed[field + "_clean"].map(lambda v: isinstance(v, list) and len(v) == len(set(v))).all()
assert not processed.tfidf_text_missing.any()
assert clean_text("  Family_with_kids / CAFÉ!  ") == "family with kids café"
assert clean_text(["Beach", None, "Food"]) == "beach food"
assert clean_text(None) == ""
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == source_hash
print("Pre-export validation passed: records preserved, features valid, source unchanged.")

Pre-export validation passed: records preserved, features valid, source unchanged.


## 9. Save processed data and audit artifacts
Save JSON rather than CSV so lists, dictionaries, booleans, and nulls retain their types. Export all flattened catalogue fields, cleaned preference arrays, TF-IDF text, structured features, and review flags. The metadata file documents preprocessing choices and the feature contract; the duplicate report keeps conflicting source values.

Rerunning replaces only these three derived artifacts, regenerating them from the source. The original file is never written. Reload and validate serialized outputs to catch type or null-conversion problems.

In [8]:
output_dir = root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
processed_path = output_dir / "india_tourism_processed.json"
duplicate_path = output_dir / "duplicate_destination_report.json"
metadata_path = output_dir / "preprocessing_metadata.json"
# pandas converts NaN to JSON null and retains native list/dictionary fields.
export_records = json.loads(processed.to_json(orient="records", force_ascii=False))
metadata = {
    "source": "data/india_tourism_dataset.json", "source_sha256": source_hash,
    "output": "data/processed/india_tourism_processed.json",
    "source_records": len(records), "processed_records": len(processed), "processed_columns": len(processed.columns),
    "duplicate_groups": int(review_keys[duplicate_mask].nunique()), "duplicate_records": int(duplicate_mask.sum()),
    "duplicate_policy": "Preserve every source entry and ID; compare and flag name + state/location groups; no merge or removal.",
    "missing_policy": "Retained unknown fields remain null; derived text uses empty strings and missing indicators; no statistical imputation.",
    "missing_value_audit": missing_audit, "budget_audit": budget_audit,
    "region_aliases": {"North East India": "Northeast India"},
    "trip_type_aliases": trip_aliases, "audience_aliases": audience_aliases,
    "text_cleaning": "NFKC, casefold, underscores to spaces, punctuation to spaces, whitespace collapse; retain Unicode letters and numbers.",
    "cost_units": "Undocumented in source; no currency or per-person assumptions applied.",
    "feature_contract": feature_contract,
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__}
}
for path, payload in [(processed_path, export_records), (duplicate_path, duplicate_reports), (metadata_path, metadata)]:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")

loaded = json.loads(processed_path.read_text(encoding="utf-8"))
assert len(loaded) == len(records)
assert [r["id"] for r in loaded] == [r["id"] for r in records]
assert all(isinstance(r["trip_types_clean"], list) and isinstance(r["ideal_for_why"], dict) for r in loaded)
assert sum(r["duplicate_destination_flag"] for r in loaded) == int(duplicate_mask.sum())
for field in numeric_fields:
    assert sum(r[field] is None for r in loaded) == int(processed[field].isna().sum())
assert all(isinstance(r["permits_required"], bool) for r in loaded)
assert json.loads(duplicate_path.read_text(encoding="utf-8")) == duplicate_reports
assert json.loads(metadata_path.read_text(encoding="utf-8"))["source_sha256"] == source_hash
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == source_hash
for path in [processed_path, duplicate_path, metadata_path]:
    print(f"Saved {path.relative_to(root)} ({path.stat().st_size:,} bytes)")
observe(f"Exported {len(loaded)} records with all source IDs and duplicate candidates preserved. "
        "Serialization checks passed and the source checksum is unchanged.")

Saved data/processed/india_tourism_processed.json (1,024,746 bytes)
Saved data/processed/duplicate_destination_report.json (89,285 bytes)
Saved data/processed/preprocessing_metadata.json (5,401 bytes)


**Observations.** Exported 100 records with all source IDs and duplicate candidates preserved. Serialization checks passed and the source checksum is unchanged.

## 10. Handoff to future recommendation work
Use `tfidf_text` as input to a future TF-IDF vectorizer, and the feature contract to select filters and ranking inputs. Preserve the reviewed raw descriptions for explanations. Resolve conflicting duplicate facts before consolidating destinations; consider grouping review candidates during evaluation and controlling repeated destinations in displayed results.

No recommendation models, statistical imputation, or feature scaling were performed. Before product use, confirm cost units, dataset provenance, and travel-sensitive information. The processed JSON is a catalogue of entries, not a verified inventory of unique destinations.